# M52 frozen-R0 FP16 evaluation gate

Evaluation only: no training and no architecture change. This runs R0 epoch 185 using selective CUDA mixed precision: FP16 autocast for compatible transformer/head operators, while backbone feature extraction, input projections feeding depth, the depth-prediction/position path, and the legacy float-only deformable-attention extension remain FP32. A mandatory dtype-instrumented one-batch CUDA smoke test runs before full validation. The notebook then applies the frozen AP, nearby-recall, localization, completeness, and provenance gates from `R0_COMPRESSION_CONTRACT.md`. Passing preserves R0; it does not qualify product safety. Run top-to-bottom on a fresh Colab GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d'); MONODETR_REPO=Path('/content/MonoDETR_M52_FP32_FEATURES')
MONODETR_COMMIT='6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti'); LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen'); DATASET_ROOT=Path('/content/monodetr_kitti_m52')
R0_SELECTION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
OFFICIAL_CHECKPOINT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/teachers/monodetr/checkpoint_best.pth')
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodetr_m52_r0_fp16_gate')
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {}); result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])


In [ ]:
# Fetch pinned sources and install tested compatibility plus FP16 evaluation patches.
if not MOBILE_REPO.exists(): run(['git','clone','https://github.com/ali-rt/mobile_adas3d.git',MOBILE_REPO])
else: run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git','clone','https://github.com/ZrrSkywalker/MonoDETR.git',MONODETR_REPO])
run(['git','fetch','--all'],cwd=MONODETR_REPO); run(['git','checkout',MONODETR_COMMIT],cwd=MONODETR_REPO)
run([sys.executable,'-m','pip','install','-q','pyyaml','scipy','opencv-python-headless','numba','scikit-image','tqdm','ninja','pandas'])
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_m52_fp16_eval.py'):
    run([sys.executable,f'scripts/{patch}','--monodetr-repo',MONODETR_REPO],cwd=MOBILE_REPO)
ops=MONODETR_REPO/'lib/models/monodetr/ops'; shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.cuda.get_device_name(0))'],cwd=MONODETR_REPO)


In [ ]:
# Create an isolated canonical Chen-split KITTI view.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names) for key,names in {'image_2':['training/image_2','training/image_02'],'label_2':['training/label_2','training/label_02'],'calib':['training/calib']}.items()}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True); (DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769


In [ ]:
# Rebuild the canonical config, verify exact R0 provenance, and prepare isolated FP16 evaluation.
for required in (R0_SELECTION,OFFICIAL_CHECKPOINT):
    if not required.is_file(): raise FileNotFoundError(required)
BASE_ROOT=Path('/content/m52_r0_base')
run([sys.executable,'scripts/prepare_monodetr_r0_reference.py','--monodetr-repo',MONODETR_REPO,'--dataset-root',DATASET_ROOT,'--official-checkpoint',OFFICIAL_CHECKPOINT,'--output-root',BASE_ROOT,'--run-name','m52_r0_base_only'],cwd=MOBILE_REPO)
BASE_CONFIG=MONODETR_REPO/'configs/monodetr_r0_vehicle_pedestrian.yaml'
run([sys.executable,'scripts/prepare_monodetr_m52_fp16_gate.py','--monodetr-repo',MONODETR_REPO,'--base-config',BASE_CONFIG,'--r0-selection',R0_SELECTION,'--output-root',OUTPUT_ROOT],cwd=MOBILE_REPO)
MANIFEST=OUTPUT_ROOT/'m52_fp16_gate_manifest.json'; manifest=json.loads(MANIFEST.read_text())
assert manifest['training_authorized'] is False and manifest['architecture_changed'] is False
assert manifest['precision']=='fp16_autocast_with_fp32_feature_depth_and_deformable_attention' and manifest['r0_epoch']==185
print(json.dumps(manifest,indent=2))


In [ ]:
# Mandatory one-batch CUDA preflight. This must pass before the 3,769-image evaluation.
SMOKE=OUTPUT_ROOT/'m52_fp16_smoke.json'
SMOKE_LOG=OUTPUT_ROOT/'colab_logs/m52_fp16_smoke.log'; SMOKE_LOG.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','scripts/smoke_test_monodetr_m52_fp16.py','--monodetr-repo',MONODETR_REPO,'--manifest',MANIFEST,'--output',SMOKE]
print('+',shlex.join(map(str,command)),'\nDurable log:',SMOKE_LOG,flush=True)
with SMOKE_LOG.open('w',encoding='utf-8',buffering=1) as log:
    process=subprocess.Popen([str(x) for x in command],cwd=MOBILE_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True); log.write(line)
    code=process.wait()
if code: raise RuntimeError(f'M52 smoke test exited {code}; full log: {SMOKE_LOG}')
smoke=json.loads(SMOKE.read_text()); assert smoke['complete'] and smoke['finite_outputs']
assert smoke['optimizer_steps']==0
assert set(smoke['backbone_feature_dtypes'])=={'torch.float32'}
assert set(smoke['projected_feature_dtypes'])=={'torch.float32'}
assert smoke['depth_predictor_dtype']=='torch.float32'
assert set(smoke['deformable_attention_kernel_dtypes'])=={'torch.float32'}
print('M52 CUDA smoke passed:',json.dumps(smoke,indent=2))


In [ ]:
# Complete FP16 inference and frozen compression-gate evaluation. Safe to rerun; complete evaluations are cached.
LOG=OUTPUT_ROOT/'colab_logs/m52_evaluation.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','scripts/evaluate_monodetr_m52_fp16_gate.py','--mobile-repo',MOBILE_REPO,'--monodetr-repo',MONODETR_REPO,'--manifest',MANIFEST,'--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR]
print('+',shlex.join(map(str,command)),'\nDurable log:',LOG,flush=True)
with LOG.open('w',encoding='utf-8',buffering=1) as log:
    process=subprocess.Popen([str(x) for x in command],cwd=MOBILE_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True); log.write(line)
    code=process.wait()
if code: raise RuntimeError(f'M52 evaluation exited {code}; full log: {LOG}')
REPORT=OUTPUT_ROOT/'m52_fp16_gate_comparison.json'; report=json.loads(REPORT.read_text())
print(json.dumps(report,indent=2))
import pandas as pd
display(pd.read_csv(OUTPUT_ROOT/'m52_fp16_gate_comparison.csv'))
print('Next compression rung authorized:',report['compression_rung_authorized'])
